# 🧬 Notebook 06: Functional Enrichment Analysis (GSEA & GO/KEGG)

## 📌 Step 1: Preparing the Ranked Gene List
Standard Over-Representation Analysis (ORA) requires a large set of strictly significant DEGs. Since our strict filtering yielded only 2 DEGs, we will utilize **Gene Set Enrichment Analysis (GSEA)**. 

GSEA does not rely on an arbitrary threshold. Instead, it evaluates microarray or RNA-Seq data at the level of gene sets. We will rank our entire dataset (3,964 genes) using a metric that combines both statistical significance and effect size: 
**Ranking Metric** = $-\log_{10}(\text{p-value}) \times \text{sign}(\log_{2}\text{FoldChange})$

In [2]:
import pandas as pd
import numpy as np
import os

print("📊 Preparing Ranked Gene List for GSEA...")

# 1. Load the full statistical results from DESeq2 (we need to save this from Notebook 5 if not done)
# Assuming you have the full res_df available, or we load it. 
# (Note: make sure you exported the full res_df to CSV in the previous notebook, or we can just reconstruct it here)
# For this script, let's load the full results from Notebook 5:
file_path = 'results/count_matrix/full_deseq2_results.csv'

try:
    res_df = pd.read_csv(file_path, index_col=0)
except FileNotFoundError:
    print(f"⚠️ Error: {file_path} not found. Make sure to export the FULL res_df from Notebook 05 first.")

# 2. Drop genes with NaN p-values to avoid calculation errors
res_clean = res_df.dropna(subset=['pvalue', 'log2FoldChange']).copy()

# 3. Calculate the ranking metric: -log10(pvalue) * sign(log2FC)
# We add a tiny number (1e-300) to p-values of 0 to avoid log(0) errors
res_clean['pvalue'] = res_clean['pvalue'].replace(0, 1e-300)
res_clean['score'] = (-np.log10(res_clean['pvalue'])) * np.sign(res_clean['log2FoldChange'])

# 4. Sort genes based on the score (descending order)
ranked_genes = res_clean[['score']].sort_values(by='score', ascending=False)

# 5. Save the ranked list for GSEA (.rnk format is standard)
os.makedirs('results/enrichment', exist_ok=True)
ranked_genes.to_csv('results/enrichment/gene_ranking.rnk', sep='\t', header=False)

print(f"✅ Successfully ranked {len(ranked_genes)} genes.")
print("\n🔝 Top 5 Up-regulated genes (by score):")
print(ranked_genes.head())
print("\n🔻 Top 5 Down-regulated genes (by score):")
print(ranked_genes.tail())

📊 Preparing Ranked Gene List for GSEA...
✅ Successfully ranked 3964 genes.

🔝 Top 5 Up-regulated genes (by score):
            score
Geneid           
Rv0341  10.024416
Rv0342   5.506273
Rv2371   4.322319
Rv1653   4.151834
Rv1656   3.801529

🔻 Top 5 Down-regulated genes (by score):
            score
Geneid           
Rv1251c -4.918554
Rv3150  -5.007467
Rv1884c -5.048456
Rv3148  -5.212229
Rv0099  -8.038105
